# EnPreNiNet — Google Colab GPU Runner (E0–E5 Ablation Matrix)

Thin orchestration notebook for reproducing **EnPreNiNet** (Ranjitha & Atham, 2026) on BDD100K night-vision data (3 classes: `car`, `person`, `traffic light`).

All core logic lives in `src/01_prepare_bdd.py` through `src/06_evaluate.py`. This notebook only mounts Google Drive, configures persistent storage symlinks, and executes CLI commands.

> **Runtime requirement:** Select `Runtime -> Change runtime type -> T4 / L4 / A100 GPU` before running.

## 1. Mount Google Drive, Clone Repository & Link Persistent Directories
Maps `data/processed`, `weights`, `runs`, and `artifacts` to `/content/drive/MyDrive/EnPreNiNet/` so checkpoints and outputs survive Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Replace REPO_URL with your GitHub repository URL if cloning remotely
!mkdir -p /content/drive/MyDrive/EnPreNiNet/data_processed /content/drive/MyDrive/EnPreNiNet/weights /content/drive/MyDrive/EnPreNiNet/runs /content/drive/MyDrive/EnPreNiNet/artifacts
!test -d /content/enpreninet-night-detection || git clone https://github.com/thiemdv/enpreninet-night-detection.git /content/enpreninet-night-detection
%cd /content/enpreninet-night-detection
!mkdir -p data
!rm -rf data/processed weights runs artifacts
!ln -s /content/drive/MyDrive/EnPreNiNet/data_processed data/processed
!ln -s /content/drive/MyDrive/EnPreNiNet/weights weights
!ln -s /content/drive/MyDrive/EnPreNiNet/runs runs
!ln -s /content/drive/MyDrive/EnPreNiNet/artifacts artifacts
!ls -la data/processed weights runs artifacts

## 2. Install Dependencies (Preserve Colab CUDA PyTorch), Clone YOLOv5 & Record Environment

In [ ]:
# Install project dependencies without overwriting Colab's prebuilt CUDA torch/torchvision
!grep -Ev '^(torch|torchvision|#|\s*$)' requirements.txt | xargs -r pip install -q
!test -d /content/yolov5 || git clone https://github.com/ultralytics/yolov5.git /content/yolov5
!grep -Ev '^(torch|torchvision|#|\s*$)' /content/yolov5/requirements.txt | xargs -r pip install -q
!nvidia-smi > artifacts/env_versions.txt
!python --version >> artifacts/env_versions.txt
!python -c "import torch, torchvision, cv2, numpy; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'torchvision:', torchvision.__version__, 'cv2:', cv2.__version__, 'numpy:', numpy.__version__)" >> artifacts/env_versions.txt
!git rev-parse HEAD >> artifacts/env_versions.txt
!cat artifacts/env_versions.txt

## 3. Prepare Raw BDD100K Dataset
Place `bdd100k_images_100k.zip` and `bdd100k_labels_release.zip` in `/content/drive/MyDrive/EnPreNiNet/raw_zips/` (or upload directly), then unpack them into `/content/raw_bdd/` on fast local Colab SSD.

In [ ]:
!mkdir -p /content/raw_bdd
!unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_images_100k.zip -d /content/raw_bdd/
!unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_labels_release.zip -d /content/raw_bdd/
!ls -la /content/raw_bdd

## 4. Stage 1 — BDD100K Night Filtering (`01_prepare_bdd.py`) & GHE Ensemble (`02_ghe_ensemble.py`)

In [ ]:
!python src/01_prepare_bdd.py --config configs/config.yaml --raw-dir /content/raw_bdd --labels-json /content/raw_bdd/bdd100k/labels/bdd100k_labels_images_train.json
!python src/02_ghe_ensemble.py --config configs/config.yaml --method standard_local_he

In [ ]:
# Optional GHE mode ablation: generate fig4_literal and fig4_normalized image variants
!python src/02_ghe_ensemble.py --config configs/config.yaml --method fig4_literal
!python src/02_ghe_ensemble.py --config configs/config.yaml --method fig4_normalized

## 5. Stage 2 — Dual-Branch GPU Training & Inference (`E0`, `E1`, `E2`, `E3`)
- **E0:** Baseline YOLOv5l on `raw` night frames
- **E1:** GHE-YOLOv5l on `standard_local_he` frames
- **E2:** Baseline Faster R-CNN ResNet-152 on `raw` night frames
- **E3:** GHE-Faster R-CNN ResNet-152 on `standard_local_he` frames

In [ ]:
# E0: Baseline YOLOv5l (raw)
!python src/03_train_infer_yolo.py --config configs/config.yaml --image-variant raw --yolo-repo /content/yolov5 --device cuda
# E1: GHE-YOLOv5l (standard_local_he)
!python src/03_train_infer_yolo.py --config configs/config.yaml --image-variant standard_local_he --yolo-repo /content/yolov5 --device cuda

In [ ]:
# E2: Baseline Faster R-CNN ResNet-152 (raw)
!python src/04_train_infer_frcnn.py --config configs/config.yaml --image-variant raw --device cuda
# E3: GHE-Faster R-CNN ResNet-152 (standard_local_he)
!python src/04_train_infer_frcnn.py --config configs/config.yaml --image-variant standard_local_he --device cuda

## 6. Stage 3 — EnPreNiNet Score-Level Fusion (`E4` vs `E5`)
- **E4:** Fusion without Global NMS (`--no-global-nms` ablation)
- **E5:** Full EnPreNiNet Fusion with class-wise Global NMS (`--global-nms`)

In [ ]:
# E4: EnPreNiNet Fusion without Global NMS
!python src/05_fusion.py --config configs/config.yaml --image-variant standard_local_he --no-global-nms --output runs/fusion_standard_local_he/fusion_standard_local_he_e4_noglobal_test_preds.pt --device cuda
# E5: EnPreNiNet Full Fusion with Global NMS
!python src/05_fusion.py --config configs/config.yaml --image-variant standard_local_he --global-nms --output runs/fusion_standard_local_he/fusion_standard_local_he_e5_full_test_preds.pt --device cuda

## 7. Full E0–E5 Evaluation & Ablation Benchmark (`06_evaluate.py`)

In [ ]:
!python src/06_evaluate.py --config configs/config.yaml --eval-split test --preds E0=runs/yolo_raw/yolo_raw_test_preds.pt E1=runs/yolo_standard_local_he/yolo_standard_local_he_test_preds.pt E2=runs/frcnn_raw/frcnn_raw_test_preds.pt E3=runs/frcnn_standard_local_he/frcnn_standard_local_he_test_preds.pt E4=runs/fusion_standard_local_he/fusion_standard_local_he_e4_noglobal_test_preds.pt E5=runs/fusion_standard_local_he/fusion_standard_local_he_e5_full_test_preds.pt
!cat artifacts/ablation_table.md